## Step 2: Clean up
**Author:** maxk678 | **Data Source:** [Garmin Data Request](https://www.garmin.com/en-US/account/datamanagement/exportdata)

This notebook takes the raw concatenated output from `01_sleep_concat.ipynb` and prepares it for analysis. 

**Input:** `sleep_concat.csv` = 800 rows, 31 columns
**Output:** `sleep_clean.csv` = 731 rows, 36 columns

## 1. Find and load CSV file

Loading the output from `01_sleep_concat.ipynb`. The data here is raw and is using Garmin field names, has values in second, displays dates as strings, and contain empty rows from nights where the watch wasn't worn. We will clean all this up below. 

In [2]:
import pandas as pd

# Update this path to point to your local processed data folder
df = pd.read_csv(r"C:\Users\maxk6\OneDrive\Desktop\DI-Connect-Wellness\Learning\data\processed\sleep_concat.csv")

print(f"Loaded: {df.shape[0]} rows, {df.shape[1]} columns")

Loaded: 800 rows, 31 columns


## 2. Drop the empty rows

69 nights have no data. This is because the watch wasn't worn those nights and Garmin exported a blank record as a placeholder. Since there is nothing to analyze in these rows, they are dropped. 

I used `dropna(subset=["calendarDate"])` instead of dropping all rows with any null. This makes sure only those missing a date (nights where the watch wasn't worn) are dropped rather than rows that are partially filled.



In [3]:
before = len(df)
df = df.dropna(subset=["calendarDate"]).reset_index(drop=True)

print(f"Dropped {before - len(df)} empty rows (watch not worn)")
print(f"Remaining: {len(df)} rows")

Dropped 69 empty rows (watch not worn)
Remaining: 731 rows


## 3. Drop columns not needed for analysis

In [4]:
df = df.drop(columns=["retro", "sleepWindowConfirmationType"])

## 4. Parse date and timestamp columns

Right now `calendarDate`, `sleepStartTimestampGMT`, and `sleepEndTimestampGMT` are plain strings. `pd.to_datetime()` converts them into proper datetime objects that pandas understands as time. This will be helpful when we want to sort, group, filter, and plot these values later. 

`utc=True` on the timestamps tells pandas these values are in UTC.  
`errors='coerce'` converts any unparseable values to `NaT` (Not a Time) instead of raising an error.

In [5]:
df["calendarDate"] = pd.to_datetime(df["calendarDate"], errors="coerce")
 
df["sleepStartTimestampGMT"] = pd.to_datetime(
    df["sleepStartTimestampGMT"], utc=True, errors="coerce"
)
df["sleepEndTimestampGMT"] = pd.to_datetime(
    df["sleepEndTimestampGMT"], utc=True, errors="coerce"
)

## 5. Derive time-based columns

These columns don't exist in the raw data. We are creating them using `calendarDate` and `sleepStartTimeStampGMT` which will help again with grouping and plotting.

- `year_month` — groups nights into months (e.g. `2024-03`) for monthly trend analysis
- `month` / `year` — useful for seasonal breakdowns
- `day_of_week` — allows weekday vs weekend comparisons
- `bed_hour` — the UTC hour sleep started; useful for bedtime analysis

`.dt` is how you access datetime properties on a pandas column after it has been parsed.

In [6]:
df["year_month"]  = df["calendarDate"].dt.to_period("M")
df["month"]       = df["calendarDate"].dt.month
df["year"]        = df["calendarDate"].dt.year
df["day_of_week"] = df["calendarDate"].dt.day_name()
df["bed_hour"]    = df["sleepStartTimestampGMT"].dt.hour

## 6. Convert sleep stages from seconds to minutes

Sleep stage durations were provided by Garmin in seconds, but minutes are more intuitive for analysis and overall more digestible overall.

Once conversion is complete, the original seconds columns are dropped. 

In [7]:
seconds_to_minutes = {
    "deepSleepSeconds":    "deep_sleep_min",
    "lightSleepSeconds":   "light_sleep_min",
    "remSleepSeconds":     "rem_sleep_min",
    "awakeSleepSeconds":   "awake_min",
    "unmeasurableSeconds": "unmeasurable_min",
}
 
for old_col, new_col in seconds_to_minutes.items():
    df[new_col] = (df[old_col] / 60).round(1)
 
df = df.drop(columns=list(seconds_to_minutes.keys()))

## 7. Calculate total sleep hours

Total sleep is the sum of deep, light, and REM sleep — it excludes time spent awake in bed, which Garmin tracks separately as `awake_min`. We use the already-converted minutes columns and divide by 60 to get hours.

In [8]:
df["total_sleep_hours"] = (
    df["deep_sleep_min"] + df["light_sleep_min"] + df["rem_sleep_min"]
) / 60
df["total_sleep_hours"] = df["total_sleep_hours"].round(2)

## 8. Create a nap flag from `napList`
The raw `napList` column contains JSON-like strings where a nap was recorded, and is null if not. Only a boolean flag is necessary for my analysis. Original column is dropped after the flag is created.

In [9]:
df["had_nap"] = df["napList"].notna()
df = df.drop(columns=["napList"])

## 9. Clean up feedback and insights label format

Feedback labels are provided by Garmin in `CONSTANT_CASE` (i.e. `POSITIVE_LONG_DEEP` or `NEGATIVE_NOT_RESTORATIVE`). To changes are made:

1. **`feedback_sentiment`** : a new column that extracts only the first word of the label (`positive`, `negative`, `neutral`)
2. **Readable labels** : labels are lowercase and underscores replaced with spaces (i.e. `POSITIVE_LONG_DEEP` becomes `positive long and deep`).


In [10]:
df["feedback_sentiment"] = (
    df["score_feedback"]
    .str.split("_")
    .str[0]
    .str.lower()
    .replace("none", "neutral")
)
 
df["score_feedback"] = (
    df["score_feedback"]
    .str.lower()
    .str.replace("_", " ", regex=False)
)
 
df["score_insight"] = (
    df["score_insight"]
    .str.lower()
    .str.replace("_", " ", regex=False)
)
 

## 10. Rename columns

Garmin's provided field names use mixed conventions. Rename everything with a consistent `snake_case` style. 

In [11]:
# Rename columns

df = df.rename(columns={
    "calendarDate":               "date",
    "sleepStartTimestampGMT":     "sleep_start_utc",
    "sleepEndTimestampGMT":       "sleep_end_utc",
    "averageRespiration":         "avg_respiration",
    "lowestRespiration":          "low_respiration",
    "highestRespiration":         "high_respiration",
    "awakeCount":                 "awake_count",
    "avgSleepStress":             "avg_sleep_stress",
    "restlessMomentCount":        "restless_moments",
    "score_overallScore":         "score_overall",
    "score_qualityScore":         "score_quality",
    "score_durationScore":        "score_duration",
    "score_recoveryScore":        "score_recovery",
    "score_deepScore":            "score_deep",
    "score_remScore":             "score_rem",
    "score_lightScore":           "score_light",
    "score_awakeningsCountScore": "score_awakenings",
    "score_awakeTimeScore":       "score_awake_time",
    "score_combinedAwakeScore":   "score_combined_awake",
    "score_restfulnessScore":     "score_restfulness",
    "score_interruptionsScore":   "score_interruptions",
    "score_feedback":             "feedback",
    "score_insight":              "insight",
})

## 11. Document remaining nulls
After dropping 69 fully empty rows, a small number of nulls remain in certain columns. These are kept intentionally since the rows they appear in contain valid, useful data. 

- **20 nights of missing REM data** : watch was worn, but REM wasn't measured. Probably due to a short or interrupted sleep
- **2-7 nights missing some score columns** : partial measurement nights where Garmin couldn't compute certain scores

These rows are excluded automatically (default of pandas) in any calculation that requires those specific columns.

In [12]:
remaining_nulls = df.isnull().sum()
remaining_nulls = remaining_nulls[remaining_nulls > 0]
print(f"\nRemaining nulls (kept intentionally):")
print(remaining_nulls)


Remaining nulls (kept intentionally):
score_overall            2
score_quality            2
score_duration           2
score_recovery           2
score_deep               7
score_rem                7
score_light              7
score_awakenings         2
score_awake_time         2
score_combined_awake     2
score_restfulness        2
score_interruptions      2
rem_sleep_min           20
total_sleep_hours       20
dtype: int64


## 12. Final Check 

In [13]:
# Final Check
print(f"\nFinal shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
print(f"\nDate range: {df['date'].min().date()}  →  {df['date'].max().date()}")


Final shape: (731, 36)
Columns: ['sleep_start_utc', 'sleep_end_utc', 'date', 'avg_respiration', 'low_respiration', 'high_respiration', 'awake_count', 'avg_sleep_stress', 'restless_moments', 'score_overall', 'score_quality', 'score_duration', 'score_recovery', 'score_deep', 'score_rem', 'score_light', 'score_awakenings', 'score_awake_time', 'score_combined_awake', 'score_restfulness', 'score_interruptions', 'feedback', 'insight', 'year_month', 'month', 'year', 'day_of_week', 'bed_hour', 'deep_sleep_min', 'light_sleep_min', 'rem_sleep_min', 'awake_min', 'unmeasurable_min', 'total_sleep_hours', 'had_nap', 'feedback_sentiment']

Date range: 2023-12-11  →  2026-02-25


## 13. Save to CSV

Saving the cleaned dataset to `sleep_clean.csv`. This file will be loaded by `03_sleep_analysis.ipynb`

In [14]:
# Save to CSV
output_path = r"C:\Users\maxk6\OneDrive\Desktop\DI-Connect-Wellness\Learning\data\processed\sleep_clean.csv"
df.to_csv(output_path, index=False)
 
print(f"\nSaved to: {output_path}")


Saved to: C:\Users\maxk6\OneDrive\Desktop\DI-Connect-Wellness\Learning\data\processed\sleep_clean.csv
